# MPC Noah's Ark — Seven-Layer Reasoning (V13)

Offline, synthetic, **not a real bounty scanner**. This notebook drives the actual GitHub `lib/noahs-ark-reasoning.mjs` implementation through the existing Node CLI, then compares selected candidates, missing evidence and adversarial cross-checks. The 239 MHA research hooks are separate from 24 native implemented MPC evaluators.

In [ ]:
from pathlib import Path
import json, subprocess, os, csv, hashlib
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "scripts" / "method-atlas-cli.mjs").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("Open this notebook from a checked-out Machine-Bug-Tool repository")
print("Repo:", ROOT)
print("No external target traffic is performed.")

In [ ]:
files = ["candidates.json", "expansion-2026-v2.json", "expansion-evidence-intent-v3.json",
         "expansion-computation-schools-v4.json", "expansion-nasa-chip-cloud-v5.json",
         "expansion-abnormal-meta-v6.json", "expansion-optical-v8.json"]
catalogs = [json.loads((ROOT/"method-atlas"/name).read_text()) for name in files]
methods = [m for c in catalogs for m in c["methods"]]
sources = [s for c in catalogs for s in c["sources"]]
relations = json.loads((ROOT/"method-atlas"/"method-relations.json").read_text())["relationships"]
assert len(methods) == 239 and len(sources) == 65 and len(relations) == 212
ids = {m["method_id"] for m in methods}
assert len(ids) == 239 and all(x["method_id"] in ids and x["related_method_id"] in ids for x in relations)
print("Source-verified metadata counts:", len(methods), "candidate methods /", len(sources), "source locators /", len(relations), "proposed links")

In [ ]:
scenarios = [
 {"name":"Synthetic ledger finality","atom":{"subject_id":"fixture:transaction","atom_id":"fixture:ledger","dimensions":["MONEY","FINALITY"],"source_refs":["fixture:owned-ledger"],"external_source_refs":[]},
  "method_receipts":[{"method_id":i,"input_state":"AVAILABLE","negative_control_state":"AVAILABLE","falsifier_state":"AVAILABLE","estimated_cost_units":cost}
                     for i,cost in [("MHA-0052",1),("MHA-0053",2),("MHA-0036",3)]]},
 {"name":"Cloud configuration — unknown inputs","atom":{"subject_id":"fixture:cloud","atom_id":"fixture:policy","dimensions":["CLOUD","AUTHORITY"],"source_refs":["fixture:config-snapshot"],"external_source_refs":[]},"method_receipts":[]},
 {"name":"Casino meter — source unavailable","atom":{"subject_id":"fixture:game","atom_id":"fixture:meter","dimensions":["MONEY","STATE"],"source_refs":[],"external_source_refs":[]},"method_receipts":[]}
]
env = os.environ.copy()
env["MPC_METHOD_ATLAS_DB"] = str(ROOT/".sites-runtime"/"noahs-ark-v13-notebook.sqlite")
def cli(command, payload=None):
    args = ["node", str(ROOT/"scripts"/"method-atlas-cli.mjs"), command]
    if payload is not None: args.append(json.dumps(payload))
    p = subprocess.run(args, cwd=ROOT, env=env, capture_output=True, text=True, timeout=120, check=True)
    return json.loads(p.stdout)
init = cli("init")
assert init["methods"] == 239
results = [(s["name"], cli("reason", {k:v for k,v in s.items() if k!="name"})) for s in scenarios]
for name, receipt in results:
    outcome=receipt["outcome_vector"]
    print(name, "|", receipt["status"], "| covered:", len(outcome["covered_dimensions"]),
          "| selected:", outcome["selected_count"], "| paired:", outcome["paired_challengers"])

In [ ]:
rows=[]
for name,receipt in results:
    v=receipt["outcome_vector"]
    rows.append({"scenario":name,"status":receipt["status"],"considered":v["catalog_methods_considered"],
      "triggered":v["structurally_triggered"],"eligible":v["structurally_eligible"],
      "selected":v["selected_count"],"pairs":v["paired_challengers"],
      "dimensions_covered":len(v["covered_dimensions"]),"dimensions_unresolved":len(v["unresolved_dimensions"]),
      "controls_ready":v["negative_controls_declared_ready"],"falsifiers_ready":v["falsifiers_declared_ready"],
      "declared_cost_units":v["declared_total_cost_units"],
      "source_authenticated":False,"method_executed":False,"security_impact_verified":False})
out = ROOT/".sites-runtime"/"noahs-ark-v13-synthetic-outcomes.csv"
out.parent.mkdir(parents=True,exist_ok=True)
with out.open("w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
print("CSV:", out)
for row in rows: print(row)

In [ ]:
for name,r in results:
    assert r["no_method_executed"] and r["no_source_authentication"] and not r["canonical_promotion"]
    assert len(r["method_consideration"]) == len(methods)
    assert len(r["reasoning_layers"]) == 7
    assert all(not p["challenger_executed"] and p["native_evidence_independence"]=="NOT_ESTABLISHED" for p in r["proposed_pairs"])
print("All 239 candidates were accounted for in each offline scenario; no method execution or security finding claimed.")

## Interpretation

Select the method with ready inputs, useful negative controls and falsifier, and coverage of unresolved typed dimensions. Pair it with a challenger where there is a declared method link. **Never use a method name, model agreement, source locator, or payoff hypothesis as authentication or proof.**

Proposed production progression: native-source readback → typed atoms → structural method shortlist → separately authorized finite evaluator → opposing controls → exact PASS checkpoint. The notebook does not sync with Google Drive or GitHub; the Drive workbook is a separate connected artifact.